In [1]:
import pandas as pd

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# I upload and read the generation and wather data, and then I merge them
df_generation = pd.read_csv('Plant_1_Generation_Data.csv', parse_dates=['DATE_TIME'])
df_weather = pd.read_csv('Plant_1_Weather_Sensor_Data.csv', parse_dates=['DATE_TIME'])
df_merged = pd.merge(df_generation, df_weather, on='DATE_TIME', how='left', suffixes=('_gen', '_weather'))
print(df_merged.head())
df_merged.to_excel('Merged_Data.xlsx', index=False)

/var/folders/sw/0d21lwk92kq5sh9w8x5brp6c0000gn/T/ipykernel_21959/1012469802.py:2: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_generation = pd.read_csv('Plant_1_Generation_Data.csv', parse_dates=['DATE_TIME'])


   DATE_TIME  PLANT_ID_gen   SOURCE_KEY_gen  DC_POWER  AC_POWER  DAILY_YIELD  \
0 2020-05-15       4135001  1BY6WEcLGh8j5v7       0.0       0.0          0.0   
1 2020-05-15       4135001  1IF53ai7Xc0U56Y       0.0       0.0          0.0   
2 2020-05-15       4135001  3PZuoBAID5Wc2HD       0.0       0.0          0.0   
3 2020-05-15       4135001  7JYdWkrLSPkdwr4       0.0       0.0          0.0   
4 2020-05-15       4135001  McdE0feGgRqW7Ca       0.0       0.0          0.0   

   TOTAL_YIELD  PLANT_ID_weather SOURCE_KEY_weather  AMBIENT_TEMPERATURE  \
0    6259559.0         4135001.0    HmiyD2TTLFNqkNe            25.184316   
1    6183645.0         4135001.0    HmiyD2TTLFNqkNe            25.184316   
2    6987759.0         4135001.0    HmiyD2TTLFNqkNe            25.184316   
3    7602960.0         4135001.0    HmiyD2TTLFNqkNe            25.184316   
4    7158964.0         4135001.0    HmiyD2TTLFNqkNe            25.184316   

   MODULE_TEMPERATURE  IRRADIATION  
0           22.857507    

In [3]:
# I ilter the dataframe for the specific Source Key, 1BY6WEcLGh8j5v7
df_specific_panel = df_merged[df_merged['SOURCE_KEY_gen'] == '1BY6WEcLGh8j5v7']

# I sort by DATE_TIME
df_specific_panel.sort_values('DATE_TIME', inplace=True)

# I create a DatetimeIndex
date_range = pd.date_range(start=df_specific_panel['DATE_TIME'].min(),
                           end=df_specific_panel['DATE_TIME'].max(),
                           freq='15T')

# I find the difference between this ideal range and the actual timestamps in the dataframe
missing_timestamps = date_range.difference(df_specific_panel['DATE_TIME'])
print(missing_timestamps)

DatetimeIndex(['2020-05-15 23:15:00', '2020-05-15 23:30:00',
               '2020-05-15 23:45:00', '2020-05-16 00:00:00',
               '2020-05-16 00:15:00', '2020-05-16 00:30:00',
               '2020-05-16 00:45:00', '2020-05-16 01:00:00',
               '2020-05-16 01:15:00', '2020-05-16 01:30:00',
               ...
               '2020-05-29 04:30:00', '2020-05-29 04:45:00',
               '2020-05-29 05:00:00', '2020-05-29 05:15:00',
               '2020-05-29 05:30:00', '2020-05-29 05:45:00',
               '2020-05-29 06:00:00', '2020-06-03 14:00:00',
               '2020-06-17 06:15:00', '2020-06-17 06:30:00'],
              dtype='datetime64[ns]', length=110, freq=None)


/var/folders/sw/0d21lwk92kq5sh9w8x5brp6c0000gn/T/ipykernel_21959/2705548495.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_specific_panel.sort_values('DATE_TIME', inplace=True)


In [4]:
# I create a new dataframe with missing timestamps
missing_df = pd.DataFrame(missing_timestamps, columns=['DATE_TIME'])

# I fill in the blanks on columns I know the value for
missing_df['PLANT_ID_gen'] = df_specific_panel['PLANT_ID_gen'].iloc[0]
missing_df['SOURCE_KEY_gen'] = '1BY6WEcLGh8j5v7'
missing_df['PLANT_ID_weather'] = df_specific_panel['PLANT_ID_weather'].iloc[0]
missing_df['SOURCE_KEY_weather'] = df_specific_panel['SOURCE_KEY_weather'].iloc[0]

# I concatenate the original dataframe with the dataframe of missing timestamps
df_complete = pd.concat([df_specific_panel, missing_df], ignore_index=True)

# I sort the complete DataFrame by DATE_TIME
df_complete.sort_values('DATE_TIME', inplace=True)
df_complete.to_excel('Complete_Data_for_Panel.xlsx', index=False)

In [5]:
# I create a new column 'imputed' and set it to 0 for all rows 

df_complete['imputed'] = 0

# these are the columns I want to impute values for, and I'll use linear interpolation for that
columns_to_interpolate = [
    'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD', 
    'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION'
]

# I run a loop that adds a value of 1 for all imputed rows so I know which they are later on
for column in columns_to_interpolate:
    mask = df_complete[column].isna()
    df_complete.loc[mask, 'imputed'] = 1

    # I Interpolate the missing values using linear interpolation
    df_complete[column] = df_complete[column].interpolate(method='linear')
df_complete.to_excel('Complete_Data_for_Panel.xlsx', index=False)

In [6]:
# Now I use time series feature engineering to add a new column called month_of_year as a value from 0 to 11.
# I replace it with the column day_of_week, predicting that the RMSE will be lower for longer time frames.
df_complete['month_of_year'] = df_complete['DATE_TIME'].dt.month

df_complete

,DATE_TIME,PLANT_ID_gen,SOURCE_KEY_gen,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD,PLANT_ID_weather,SOURCE_KEY_weather,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION,imputed,month_of_year
0,2020-05-15 00:00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6259559.0,4135001.0,HmiyD2TTLFNqkNe,25.184316,22.857507,0.0,0,5
1,2020-05-15 00:15:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6259559.0,4135001.0,HmiyD2TTLFNqkNe,25.084589,22.761668,0.0,0,5
2,2020-05-15 00:30:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6259559.0,4135001.0,HmiyD2TTLFNqkNe,24.935753,22.592306,0.0,0,5
3,2020-05-15 00:45:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6259559.0,4135001.0,HmiyD2TTLFNqkNe,24.846130,22.360852,0.0,0,5
4,2020-05-15 01:00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6259559.0,4135001.0,HmiyD2TTLFNqkNe,24.621525,22.165423,0.0,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3149,2020-06-17 22:45:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,5521.0,6485319.0,4135001.0,HmiyD2TTLFNqkNe,22.150570,21.480377,0.0,0,6
3150,2020-06-17 23:00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,5521.0,6485319.0,4135001.0,HmiyD2TTLFNqkNe,22.129816,21.389024,0.0,0,6
3151,2020-06-17 23:15:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,5521.0,6485319.0,4135001.0,HmiyD2TTLFNqkNe,22.008275,20.709211,0.0,0,6
3152,2020-06-17 23:30:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,5521.0,6485319.0,4135001.0,HmiyD2TTLFNqkNe,21.969495,20.734963,0.0,0,6


In [7]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# I calculate a naive baseline

X = df_complete[['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION','month_of_year']]
y = df_complete['AC_POWER']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

y_train_median = y_train.median()
y_naive_pred = np.full(shape=y_test.shape, fill_value=y_train_median)

mae_naive = mean_absolute_error(y_test, y_naive_pred)
print('Naive Baseline MAE:', mae_naive)

mse_naive = mean_squared_error(y_test, y_naive_pred)
print('Naive Baseline MSE:', mse_naive)

rmse_naive = np.sqrt(mse_naive)
print('Naive Baseline RMSE:', rmse_naive)

Naive Baseline MAE: 266.0781670367449
Naive Baseline MSE: 183122.3930413372
Naive Baseline RMSE: 427.9280232017263


In [9]:
# Now I repeat the previous steps to form the rest of the models

In [10]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

mae_linear = mean_absolute_error(y_test, y_pred)
print('Linear Regression MAE:', mae_linear)

mse_linear = mean_squared_error(y_test, y_pred)
print('Linear Regression MSE:', mse_linear)

rmse_linear = np.sqrt(mse_linear)
print('Linear Regression RMSE:', rmse_linear)

Linear Regression MAE: 33.43048276294528
Linear Regression MSE: 10531.156046815953
Linear Regression RMSE: 102.62142099394235


In [11]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_rf_pred = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_rf_pred)
print('Random Forest MAE:', mae_rf)

mse_rf = mean_squared_error(y_test, y_rf_pred)
print('Random Forest MSE:', mse_rf)

rmse_rf = np.sqrt(mse_rf)
print('Random Forest RMSE:', rmse_rf)

Random Forest MAE: 31.09488093755319
Random Forest MSE: 9578.314952646118
Random Forest RMSE: 97.86886610483499


In [12]:
from sklearn.svm import SVR

svr_model = SVR(kernel='rbf', C=1.0, epsilon=0.1)

svr_model.fit(X_train, y_train)

y_svr_pred = svr_model.predict(X_test)

mae_svr = mean_absolute_error(y_test, y_svr_pred)
print('SVR MAE:', mae_svr)

mse_svr = mean_squared_error(y_test, y_svr_pred)
print('SVR MSE:', mse_svr)

rmse_svr = np.sqrt(mse_svr)
print('SVR RMSE:', rmse_svr)

SVR MAE: 95.70210366201964
SVR MSE: 24843.97791933186
SVR RMSE: 157.6197256669731


In [13]:
from sklearn.ensemble import GradientBoostingRegressor

gbr_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

gbr_model.fit(X_train, y_train)

y_gbr_pred = gbr_model.predict(X_test)

mae_gbr = mean_absolute_error(y_test, y_gbr_pred)
print('Gradient Boosting Regressor MAE:', mae_gbr)

mse_gbr = mean_squared_error(y_test, y_gbr_pred)
print('Gradient Boosting Regressor MSE:', mse_gbr)

rmse_gbr = np.sqrt(mse_gbr)
print('Gradient Boosting Regressor RMSE:', rmse_gbr)

Gradient Boosting Regressor MAE: 31.09847022916562
Gradient Boosting Regressor MSE: 9711.440815135322
Gradient Boosting Regressor RMSE: 98.54664284051142


In [14]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_model = KNeighborsRegressor(n_neighbors=5)

knn_model.fit(X_train_scaled, y_train)

y_knn_pred = knn_model.predict(X_test_scaled)

mae_knn = mean_absolute_error(y_test, y_knn_pred)
print('KNN Regressor MAE:', mae_knn)

mse_knn = mean_squared_error(y_test, y_knn_pred)
print('KNN Regressor MSE:', mse_knn)

rmse_knn = np.sqrt(mse_knn)
print('KNN Regressor RMSE:', rmse_knn)

KNN Regressor MAE: 31.048110179107532
KNN Regressor MSE: 8440.796568714151
KNN Regressor RMSE: 91.87380784921321


In [15]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_model = KNeighborsRegressor(n_neighbors=5)

knn_model.fit(X_train_scaled, y_train)

y_knn_pred = knn_model.predict(X_test_scaled)

mae_knn = mean_absolute_error(y_test, y_knn_pred)
print('KNN Regressor MAE:', mae_knn)

mse_knn = mean_squared_error(y_test, y_knn_pred)
print('KNN Regressor MSE:', mse_knn)

rmse_knn = np.sqrt(mse_knn)
print('KNN Regressor RMSE:', rmse_knn)

KNN Regressor MAE: 31.048110179107532
KNN Regressor MSE: 8440.796568714151
KNN Regressor RMSE: 91.87380784921321


In [16]:
# Finally, I show the performance of all mdoels using MAE, MSE and RMSE

results = {
    'Model': ['Naive Baseline', 'Linear Regression', 'Random Forest', 'SVR', 'Gradient Boosting', 'KNN'],
    'MAE': [mae_naive, mae_linear, mae_rf, mae_svr, mae_gbr, mae_knn],
    'MSE': [mse_naive, mse_linear, mse_rf, mse_svr, mse_gbr, mse_knn],
    'RMSE': [rmse_naive, rmse_linear, rmse_rf, rmse_svr, rmse_gbr, rmse_knn]
}

results_df = pd.DataFrame(results)

results_df.set_index('Model', inplace=True)

results_df_sorted_by_rmse = results_df.sort_values(by='RMSE', ascending=True)

print(results_df_sorted_by_rmse)

                          MAE            MSE        RMSE
Model                                                   
KNN                 31.048110    8440.796569   91.873808
Random Forest       31.094881    9578.314953   97.868866
Gradient Boosting   31.098470    9711.440815   98.546643
Linear Regression   33.430483   10531.156047  102.621421
SVR                 95.702104   24843.977919  157.619726
Naive Baseline     266.078167  183122.393041  427.928023


In [17]:
#RMSE is slightly lower than its original value (91.89). Therefore, we did a better prediction.